#### Wczytanie i połączenie plików grid

In [2]:
# importy potrzebnych pakietów oraz określenie głównej ścieżki projektu

import xarray as xr
import cfgrib
import glob
xr.set_options(arithmetic_join='outer')
import os
import numpy as np

main_path = r'C:\Users\bagaw\Desktop\sem5\ML\climat_project'

In [2]:
# Funkcja wczytująca i łącząca pliki z podanej ścieżki
# oraz przycinająca je do wybranych współrzędnych

def process_and_stack_gribs(file_list, lat_range, lon_range):
    all_datasets = []

    for file_path in file_list:
        print(f"Przetwarzanie pliku: {file_path}...")
        try:
            dss = cfgrib.open_datasets(file_path)
        except Exception as e:
            print(f"Błąd wczytywania {file_path}: {e}")
            continue

        cleaned_groups = []
        for ds in dss:
            lon_name = [k for k in ds.coords if 'lon' in k.lower()][0]
            lat_name = [k for k in ds.coords if 'lat' in k.lower()][0]

            if ds[lon_name].max() > 180:
                ds = ds.assign_coords({lon_name: 
                                           (((ds[lon_name] + 180) % 360) - 180)})
            
            lat_slice = slice(max(lat_range), min(lat_range)) \
                if ds[lat_name][0] > ds[lat_name][-1] \
                else slice(min(lat_range), max(lat_range))
            ds = ds.sel({lat_name: lat_slice, lon_name: 
                slice(min(lon_range), max(lon_range))})

            if 'step' in ds.dims:
                ds = ds.stack(combined_time=('time', 'step'))
                ds = ds.drop_vars('time').rename({'combined_time': 'time'})
                ds['time'] = ds.valid_time
                ds = ds.drop_vars('valid_time')
            
            cleaned_groups.append(ds)
        
        file_ds = xr.merge(cleaned_groups, compat='override', join='outer')
        all_datasets.append(file_ds)

    combined_all = xr.concat(all_datasets, dim='time').sortby('time')
    combined_all = combined_all.drop_duplicates('time')
    return combined_all

In [4]:
# Tworzy listę wszystkich plików .grib w danym folderze
path_1 = os.path.join(main_path, 
                      r'data\before_2010\single_levels_portugal*.grib')
files_1 = glob.glob(path_1)
path_2 = os.path.join(main_path, 
                      r'data\after_2010\single_levels_portugal*.grib')
files_2 = glob.glob(path_2)

before_10_path = os.path.join(main_path, 'end_result_b10.nc')
#after_10_path  = os.path.join(main_path, 'end_result_a10.nc')
           
final_ds_1 = process_and_stack_gribs(files_1, (39.5,41.5), (-7.75,-7.0))
#final_ds_2 = process_and_stack_gribs(files_2, (39.5,41.5), (-7.8,-6.8))

final_ds_1.to_netcdf(before_10_path)
#final_ds_2.to_netcdf(after_10_path)

Przetwarzanie pliku: C:\Users\bagaw\Desktop\sem5\ML\climat_project\data\before_2010\single_levels_portugal_1998_2002_aligned.grib...


C:\Users\bagaw\anaconda3\envs\climat_project\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
C:\Users\bagaw\anaconda3\envs\climat_project\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)


Przetwarzanie pliku: C:\Users\bagaw\Desktop\sem5\ML\climat_project\data\before_2010\single_levels_portugal_1999_2002_aligned.grib...


C:\Users\bagaw\anaconda3\envs\climat_project\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
C:\Users\bagaw\anaconda3\envs\climat_project\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)


Przetwarzanie pliku: C:\Users\bagaw\Desktop\sem5\ML\climat_project\data\before_2010\single_levels_portugal_345.grib...


...

C:\Users\bagaw\anaconda3\envs\climat_project\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
C:\Users\bagaw\anaconda3\envs\climat_project\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)


Przetwarzanie pliku: C:\Users\bagaw\Desktop\sem5\ML\climat_project\data\before_2010\single_levels_portugal_9.grib...


C:\Users\bagaw\anaconda3\envs\climat_project\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
C:\Users\bagaw\anaconda3\envs\climat_project\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)


Przetwarzanie pliku: C:\Users\bagaw\Desktop\sem5\ML\climat_project\data\before_2010\single_levels_portugal_95_98.grib...


C:\Users\bagaw\anaconda3\envs\climat_project\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
C:\Users\bagaw\anaconda3\envs\climat_project\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)


#### Nadanie danym pożądanej struktury oraz złączenie ich w jeden plik

In [4]:
def open_function(path, save_path):
    # nadanie struktury: czas, szerokość, długość
    with xr.open_dataset(path, engine='h5netcdf') as source:
        data = source.load()
    ds = data.transpose('time', 'latitude', 'longitude')

    # czyszczenie metadanych ze zbędnych atrybutów GRIB
    for var in ds.data_vars:
        bad_attrs = [a for a in ds[var].attrs if a.startswith('GRIB_')]
        for attr in bad_attrs:
            del ds[var].attrs[attr]
    
    # usuwanie zbędnych zmiennych (ich wartości nie są znaczące 
    # dla dalszej analizy lub zostały zastapione przez inne kolumny)
    ds = ds.drop_vars(['step', 'surface', 'depthBelowLandLayer', 
                       'number', 'lsm', 'valid_time' ], errors='ignore')
    ds.to_netcdf(save_path)

In [5]:
save_path_b10  = os.path.join(main_path, 'end_data_b10.nc')
save_path_a10  = os.path.join(main_path, 'end_data_a10.nc')

before_10_path = os.path.join(main_path, 'end_result_b10.nc')
after_10_path  = os.path.join(main_path, 'end_result_a10.nc')
open_function(before_10_path, save_path_b10)
open_function(after_10_path, save_path_a10)

In [6]:
ds_b10 = xr.open_dataset(save_path_b10)
ds_a10 = xr.open_dataset(save_path_a10)

ds_final = xr.concat([ds_b10, ds_a10], dim='time')

print(f"Nowy wymiar czasu: {len(ds_final.time)}")
print(ds_final)



Nowy wymiar czasu: 43832
<xarray.Dataset> Size: 73MB
Dimensions:    (time: 43832, latitude: 8, longitude: 4)
Coordinates:
  * time       (time) datetime64[ns] 351kB 1995-01-01 ... 2024-12-31T18:00:00
  * latitude   (latitude) float64 64B 41.4 41.15 40.9 40.65 ... 40.15 39.9 39.65
  * longitude  (longitude) float64 32B -7.75 -7.5 -7.25 -7.0
Data variables: (12/13)
    swvl1      (time, latitude, longitude) float32 6MB 0.4307 0.4328 ... 0.2866
    swvl2      (time, latitude, longitude) float32 6MB 0.4314 0.4319 ... 0.2939
    swvl3      (time, latitude, longitude) float32 6MB 0.4019 0.3734 ... 0.3046
    swvl4      (time, latitude, longitude) float32 6MB 0.3545 0.324 ... 0.2936
    cape       (time, latitude, longitude) float32 6MB 16.25 19.05 ... 0.0 0.0
    u10        (time, latitude, longitude) float32 6MB 1.121 1.396 ... -1.334
    ...         ...
    t2m        (time, latitude, longitude) float32 6MB 280.4 280.6 ... 280.5
    d2m        (time, latitude, longitude) float32 6MB 279.7 

In [7]:
# zmiana jednostek na bardziej intuicyjne do analizy

# temperatura: Kelwiny -> Celsjusz
ds_final[['t2m', 'd2m']] = ds_final[['t2m', 'd2m']] - 273.15
ds_final['t2m'].attrs['units'] = ds_final['d2m'].attrs['units'] = '°C'

# opady i parowanie : m -> mm
ds_final[['tp', 'sf', 'e']] = ds_final[['tp', 'sf', 'e']]*1000
ds_final['tp'].attrs['units'] = (
    ds_final['sf'].attrs)['units'] = ds_final['e'].attrs['units'] = 'mm'

# Obliczanie wypadkowej prędkości wiatru
ds_final['wind_speed'] = (ds_final['u10']**2 + ds_final['v10']**2)**0.5
ds_final['wind_speed'].attrs['units'] = 'm s**-1'
ds_final['wind_speed'].attrs['long_name'] = '10 metre wind speed'

# Obliczanie kierunku, z którego wieje wiatr 
ds_final['wind_dir'] = (np.rad2deg(
    np.arctan2(ds_final['u10'], ds_final['v10'])) + 180) % 360
ds_final['wind_dir'].attrs['units'] = 'degrees'
ds_final['wind_dir'].attrs['long_name'] = '10 metre wind direction'

ds_final = ds_final.drop_vars(['u10', 'v10'])

# promieniowanie: J/m2 -> MJ/m2
ds_final['ssrd'] = ds_final['ssrd'] / 1e6
ds_final['ssrd'].attrs['units'] = 'MJ m**-2'

# zmiana opisu atrybutu czasu (po zestackowaniu ze step w kolumnie time
# znajduje sie czas rzeczywisty, nie czas prognozy jak wcześniej)

ds_final.time.attrs['long_name'] = ds_final.time.attrs['standard_name'] ='time'

print(ds_final)

<xarray.Dataset> Size: 73MB
Dimensions:     (time: 43832, latitude: 8, longitude: 4)
Coordinates:
  * time        (time) datetime64[ns] 351kB 1995-01-01 ... 2024-12-31T18:00:00
  * latitude    (latitude) float64 64B 41.4 41.15 40.9 ... 40.15 39.9 39.65
  * longitude   (longitude) float64 32B -7.75 -7.5 -7.25 -7.0
Data variables: (12/13)
    swvl1       (time, latitude, longitude) float32 6MB 0.4307 0.4328 ... 0.2866
    swvl2       (time, latitude, longitude) float32 6MB 0.4314 0.4319 ... 0.2939
    swvl3       (time, latitude, longitude) float32 6MB 0.4019 0.3734 ... 0.3046
    swvl4       (time, latitude, longitude) float32 6MB 0.3545 0.324 ... 0.2936
    cape        (time, latitude, longitude) float32 6MB 16.25 19.05 ... 0.0 0.0
    t2m         (time, latitude, longitude) float32 6MB 7.284 7.437 ... 7.322
    ...          ...
    sf          (time, latitude, longitude) float32 6MB 0.0 0.0 0.0 ... 0.0 0.0
    ssrd        (time, latitude, longitude) float32 6MB 0.0 0.0 ... 0.003021
  

In [8]:
# zapis ostateczny pliku
end_path = os.path.join(main_path, 'full_data_2.nc')
ds_final.to_netcdf(end_path)